# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [1]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on GitHub we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [3]:
# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [4]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [5]:
cur.execute("SELECT * FROM eda.king_county_house_sales LIMIT 10")
cur.fetchall()

[(datetime.date(2014, 10, 13), 221900.0, 7129300520, 1),
 (datetime.date(2014, 12, 9), 538000.0, 6414100192, 2),
 (datetime.date(2015, 2, 25), 180000.0, 5631500400, 3),
 (datetime.date(2014, 12, 9), 604000.0, 2487200875, 4),
 (datetime.date(2015, 2, 18), 510000.0, 1954400510, 5),
 (datetime.date(2014, 5, 12), 1230000.0, 7237550310, 6),
 (datetime.date(2014, 6, 27), 257500.0, 1321400060, 7),
 (datetime.date(2015, 1, 15), 291850.0, 2008000270, 8),
 (datetime.date(2015, 4, 15), 229500.0, 2414600126, 9),
 (datetime.date(2015, 3, 12), 323000.0, 3793500160, 10)]

With `conn.close()` you can close the connection again.

In [6]:
# close the connection
conn.close()

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [7]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

In [ ]:
# import the data into a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales LIMIT 10"
df_psycopg = pd.read_sql(query_string, conn)
print("done")

In [10]:
# close the connection
conn.close()

In [11]:
df_psycopg.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


In [12]:
# export the data to a csv-file
df_psycopg.to_csv("data/eda.csv", index=False)

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database string (a link that includes every information we entered in the conn object)

In [13]:
from sqlalchemy import create_engine

# read the database string from the .env
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [17]:
# import the data to a pandas dataframe
# "SELECT * FROM eda.king_county_house_sales"
query_string = "SELECT * FROM eda.king_county_house_sales kchs INNER JOIN eda.king_county_house_details kchd ON kchs.house_id = kchd.id"
df_sqlalchemy = pd.read_sql(query_string, db)

In [15]:
df_sqlalchemy.head()

,date,price,house_id,id,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,7129300520,3.0,1.00,1180.0,5650.0,1.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,6414100192,3.0,2.25,2570.0,7242.0,2.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,5631500400,2.0,1.00,770.0,10000.0,1.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,2487200875,4.0,3.00,1960.0,5000.0,1.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,1954400510,3.0,2.00,1680.0,8080.0,1.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [18]:
# export the data to a csv-file
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [19]:
# import the data from a csv-file
df_import = pd.read_csv("data/eda.csv")

In [20]:
df_import.head()

,date,price,house_id,id,id.1,bedrooms,bathrooms,sqft_living,sqft_lot,floors,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,7129300520,3.0,1.00,1180.0,5650.0,1.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,6414100192,3.0,2.25,2570.0,7242.0,2.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,5631500400,2.0,1.00,770.0,10000.0,1.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,2487200875,4.0,3.00,1960.0,5000.0,1.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,1954400510,3.0,2.00,1680.0,8080.0,1.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [21]:
df_import.shape

(21597, 23)

In [22]:
df_import.drop(columns=["id.1", "id"], inplace=True)
df_import.head()

,date,price,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [23]:
df1 = df_import

In [24]:
df1['is_renovated'] = df1['yr_renovated'] > 0
df1['price_per_sqft'] = df1['price'] / df1['sqft_living']

old_homes = df1[df1['yr_built'] < 1980]
print(old_homes.groupby('is_renovated')['price_per_sqft'].mean())

is_renovated
False    277.620228
True     321.741619
Name: price_per_sqft, dtype: float64


In [25]:
df1.head(10)

,date,price,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_per_sqft
0,2014-10-13,221900.0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,...,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,False,188.050847
1,2014-12-09,538000.0,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,...,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0,True,209.338521
2,2015-02-25,180000.0,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,...,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,False,233.766234
3,2014-12-09,604000.0,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,...,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,False,308.163265
4,2015-02-18,510000.0,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,...,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,False,303.571429
5,2014-05-12,1230000.0,7237550310,4.0,4.50,5420.0,101930.0,1.0,0.0,0.0,...,1530.0,2001,0.0,98053,47.6561,-122.005,4760.0,101930.0,False,226.937269
6,2014-06-27,257500.0,1321400060,3.0,2.25,1715.0,6819.0,2.0,0.0,0.0,...,NaN,1995,0.0,98003,47.3097,-122.327,2238.0,6819.0,False,150.145773
7,2015-01-15,291850.0,2008000270,3.0,1.50,1060.0,9711.0,1.0,0.0,NaN,...,0.0,1963,0.0,98198,47.4095,-122.315,1650.0,9711.0,False,275.330189
8,2015-04-15,229500.0,2414600126,3.0,1.00,1780.0,7470.0,1.0,0.0,0.0,...,730.0,1960,0.0,98146,47.5123,-122.337,1780.0,8113.0,False,128.932584
9,2015-03-12,323000.0,3793500160,3.0,2.50,1890.0,6560.0,2.0,0.0,0.0,...,0.0,2003,0.0,98038,47.3684,-122.031,2390.0,7570.0,False,170.899471


In [26]:
zip_stats = df1.groupby('zipcode').agg(
    avg_price_per_sqft=('price_per_sqft', 'mean'),
    avg_grade=('grade', 'mean'),
    count=('price', 'size')
)#.sort_values('avg_price_per_sqft', ascending=False)

print(zip_stats.head(15))

         avg_price_per_sqft  avg_grade  count
zipcode                                      
98001            151.347966   7.296399    361
98002            151.174091   6.693467    199
98003            157.113414   7.542857    280
98004            475.609615   8.687697    317
98005            314.966998   8.488095    168
98006            299.149268   8.795181    498
98007            290.090234   7.964539    141
98008            301.745997   7.653710    283
98010            210.095356   7.400000    100
98011            225.996296   7.774359    195
98014            223.084512   7.387097    124
98019            203.007336   7.510526    190
98022            182.106295   7.175966    233
98023            148.921543   7.575150    499
98024            252.327749   7.612500     80


In [27]:
zip_stats.shape

(70, 3)

In [28]:
df1['date'] = pd.to_datetime(df1['date'])
df1['sale_month'] = df1['date'].dt.month

monthly_stats = df1.groupby('sale_month').agg(
    avg_price=('price', 'mean'),
    avg_price_per_sqft=('price_per_sqft', 'mean'),
    num_sales=('price', 'size')
)
print(monthly_stats)

                avg_price  avg_price_per_sqft  num_sales
sale_month                                              
1           525963.251534          256.773212        978
2           508520.051323          259.770095       1247
3           544057.683200          273.811231       1875
4           562215.615074          278.835612       2229
5           550849.746893          269.110846       2414
6           557534.318182          264.809163       2178
7           544892.161013          259.691581       2211
8           536655.212481          259.889913       1939
9           529723.517787          259.378960       1771
10          539439.447228          262.402144       1876
11          522359.903478          258.495379       1409
12          524799.902041          254.685079       1470


In [29]:
df1

,date,price,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_per_sqft,sale_month
0,2014-10-13,221900.0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,...,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,False,188.050847,10
1,2014-12-09,538000.0,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,...,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0,True,209.338521,12
2,2015-02-25,180000.0,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,...,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,False,233.766234,2
3,2014-12-09,604000.0,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,...,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,False,308.163265,12
4,2015-02-18,510000.0,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,...,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,False,303.571429,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,2014-05-21,360000.0,263000018,3.0,2.50,1530.0,1131.0,3.0,0.0,0.0,...,2009,0.0,98103,47.6993,-122.346,1530.0,1509.0,False,235.294118,5
21593,2015-02-23,400000.0,6600060120,4.0,2.50,2310.0,5813.0,2.0,0.0,0.0,...,2014,0.0,98146,47.5107,-122.362,1830.0,7200.0,False,173.160173,2
21594,2014-06-23,402101.0,1523300141,2.0,0.75,1020.0,1350.0,2.0,0.0,0.0,...,2009,0.0,98144,47.5944,-122.299,1020.0,2007.0,False,394.216667,6
21595,2015-01-16,400000.0,291310100,3.0,2.50,1600.0,2388.0,2.0,NaN,0.0,...,2004,0.0,98027,47.5345,-122.069,1410.0,1287.0,False,250.000000,1


In [30]:
# find houses sold more than once
sale_counts = df1['house_id'].value_counts()
multi_sold = df1[df1['house_id'].isin(sale_counts[sale_counts > 1].index)].copy()
multi_sold['date'] = pd.to_datetime(multi_sold['date'])
multi_sold = multi_sold.sort_values(['house_id', 'date'])

# 1. price change between first and last sale
first_last = multi_sold.groupby('house_id').agg(
    first_price=('price', 'first'),
    last_price=('price', 'last'),
    first_date=('date', 'first'),
    last_date=('date', 'last')
)
first_last['price_change'] = first_last['last_price'] - first_last['first_price']
print(first_last['price_change'].mean())

136567.61931818182


In [31]:
# 2. which zipcodes have most repeat sales
print(multi_sold['zipcode'].value_counts().head(20))

# 3. grade/condition of repeat-sold houses vs rest
print(multi_sold['grade'].mean(), df1['grade'].mean())
print(multi_sold['condition'].mean(), df1['condition'].mean())

zipcode
98133    18
98118    18
98055    16
98006    16
98023    14
98146    14
98115    14
98125    14
98074    12
98106    10
98198    10
98117    10
98168     9
98003     8
98166     8
98178     8
98058     8
98155     8
98001     6
98038     6
Name: count, dtype: int64
7.073654390934844 7.657915451220076
3.2974504249291785 3.4098254387183404
